# Hands-on Beispiel LLM (1)

### 1. Baseline - Juristische Fragen an ein untrainiertes Modell (Lokale LLM)
In diesem Abschnitt laden wir das Modell `dbmdz/german-gpt2` und stellen ihm zwei juristische Fragen zum AI Act.Da das Modell noch nicht mit AI Act-spezifischen Texten feingetunt wurde, sind die Antworten vermutlich unzureichend.


In [1]:
# falls noch nicht installiert 

import sys
# !{sys.executable} -m pip install transformers datasets
# !{sys.executable} -m pip install 'accelerate>=0.26.0'

In [2]:
# Importiere die benötigten Bibliotheken
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# Lade das deutschsprachige Modell und den zugehörigen Tokenizer

model_name = "dbmdz/german-gpt2" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model = model.bfloat16()

# Konfiguriere pad_token_id im Modell 
# (braucht man, wenn das Modell noch nicht standardmäßig für den Umgang mit dem Padding-Token eingestellt ist)
model.config.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.eval()

# Da GPT-2-Modelle oft keinen expliziten Padding-Token besitzen, setzen wir hier den EOS-Token als Padding-Token.
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: dbmdz/german-gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def ask_question(prompt):
    # Tokenisiere den Input inklusive Attention-Maske
    encoded = tokenizer(prompt, return_tensors="pt")
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    
    # Generiere eine Antwort unter Verwendung der Attention-Maske
    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=150,
        temperature=0.7,
        do_sample=True
    )
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text

In [5]:
# Liste juristischer Fragen, bei denen Wissen aus dem AI Act abgefragt wird
questions = [
    "Welche Anforderungen stellt der AI Act an Hochrisiko-KI-Systeme?",
    "Was versteht man unter Transparenz gemäß dem AI Act?"
]

In [6]:
# Fragen stellen und Antworten ausgeben
print("=== llm-1: Untrainiertes Modell ===\n")
for q in questions:
    print("Frage:", q)
    print() 
    answer = ask_question(q)
    print("Antwort:", answer)
    print("-" * 500)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== llm-1: Untrainiertes Modell ===

Frage: Welche Anforderungen stellt der AI Act an Hochrisiko-KI-Systeme?



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Antwort: Welche Anforderungen stellt der AI Act an Hochrisiko-KI-Systeme?
Wie kann der AI Act zur Erreichung der Ziele des neuen AI Act beitragen?
Wie können die AI-Anforderungen erfüllt werden, um die Sicherheit Ihres Unternehmens zu verbessern?
Wie können Sie der Gefahr von IT-Angriffen vorbeugen?
Wie kann ein Sicherheitsrisiko von Ihrem Unternehmen minimiert werden?
Wie kann ein Sicherheitsprotokoll von Daten oder Systemen, die mit Sicherheitstechnologien arbeiten, die Integrität Ihres Unternehmens schützen?
Wie kann Sicherheitsbewusstsein auf Ihre IT-Systeme und Unternehmen und die damit verbundenen Risiken gesenkt werden?
Wie kann ein Sicherheits-Management von Ihrem Unternehmen einen Mehrwert für Ihre Organisation bieten?
Wie kann ein Unternehmen ein Unternehmen in eine IT
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------